In [2]:
import cv2
import numpy as np
import mediapipe as mp
import time

mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils

idx = 5

Videolist = ["Video/SideView_1.mp4","Video/SideView_2.mp4","Video/SideView_3.mp4","Video/SideView_4.mp4",
             "Video/RearView_1.mp4","Video/RearView_2.mp4","Video/RearView_3.mp4", "Video/OverheadView_1.mp4"]

SetVideolist = [
    "Video/NewVideo/SetSideView_1.mp4","Video/NewVideo/SetSideView_2.mp4","Video/NewVideo/SetSideView_3.mp4","Video/NewVideo/SetSideView_4.mp4",
    "Video/NewVideo/SetRearView_1.mp4","Video/NewVideo/SetRearView_2.mp4","Video/NewVideo/SetRearView_3.mp4",
    "Video/NewVideo/SetOverheadView_1.mp4"
]

cap = cv2.VideoCapture(Videolist[idx])

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

out_size = (width * 2, height)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(SetVideolist[idx], fourcc, fps, out_size)


with mp_pose.Pose(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7,
    static_image_mode=False, 
    model_complexity=2
    ) as pose :
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        skeleton_img = np.zeros_like(frame)
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = pose.process(image_rgb)

        if res.pose_landmarks:
            mp_draw.draw_landmarks(frame, res.pose_landmarks, mp_pose.POSE_CONNECTIONS)
            mp_draw.draw_landmarks(skeleton_img, res.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        
        # image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        cv2.namedWindow("MediaPipe", cv2.WINDOW_NORMAL)
        cv2.resizeWindow("MediaPipe", 1280, 720)
        combined = np.hstack((frame, skeleton_img))
        cv2.imshow("MediaPipe",combined)

        out.write(combined)
        
        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
out.release()
cv2.destroyAllWindows()